In [0]:
%run ./common

In [0]:
from datetime import datetime, timezone
from delta.tables import DeltaTable

spark.conf.set('spark.sql.session.timeZone', 'UTC')

G = f'{CATALOG}.gold'   # CATALOG comes from ./common


def run(sql):
    """Run one SQL statement and return the DataFrame."""
    return spark.sql(sql)


run(f'CREATE SCHEMA IF NOT EXISTS {G}')
print('Schema ready:', G)

In [0]:
run(f'''
CREATE TABLE IF NOT EXISTS {G}.dim_date (
  date_key          INT NOT NULL,
  full_date         DATE,
  year              INT,
  quarter           INT,
  month_num         INT,
  month_name        STRING,
  year_month        STRING,
  month_start_date  DATE,
  week_of_year      INT,
  day_of_week_num   INT,
  day_name          STRING,
  is_weekend        BOOLEAN,
  CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
) USING DELTA
''')

run(f'''
CREATE TABLE IF NOT EXISTS {G}.dim_repository (
  repository_key    BIGINT NOT NULL,
  repository_name   STRING NOT NULL,
  owner             STRING,
  repo_name         STRING,
  description       STRING,
  language          STRING,
  default_branch    STRING,
  html_url          STRING,
  created_at        TIMESTAMP,
  archived          BOOLEAN,
  load_timestamp    TIMESTAMP,
  CONSTRAINT pk_dim_repository PRIMARY KEY (repository_key)
) USING DELTA
''')

run(f'''
CREATE TABLE IF NOT EXISTS {G}.dim_contributor (
  contributor_key     BIGINT NOT NULL,
  account_type        STRING,
  is_bot              BOOLEAN,
  first_activity_date DATE,
  load_timestamp      TIMESTAMP,
  CONSTRAINT pk_dim_contributor PRIMARY KEY (contributor_key)
) USING DELTA
''')
print('Dimensions ready')

In [0]:
FK_REPO = f'FOREIGN KEY (repository_key) REFERENCES {G}.dim_repository(repository_key)'
FK_CONTRIB = f'FOREIGN KEY (contributor_key) REFERENCES {G}.dim_contributor(contributor_key)'


def fk_date(column):
    return f'FOREIGN KEY ({column}) REFERENCES {G}.dim_date(date_key)'


run(f'''
CREATE TABLE IF NOT EXISTS {G}.fact_commits (
  record_key        STRING NOT NULL,
  commit_sha        STRING,
  repository_key    BIGINT NOT NULL,
  contributor_key   BIGINT NOT NULL,
  commit_date_key   INT NOT NULL,
  commit_timestamp  TIMESTAMP,
  message_length    INT,
  commit_count      INT,
  batch_id          STRING,
  load_timestamp    TIMESTAMP,
  CONSTRAINT pk_fact_commits PRIMARY KEY (record_key),
  CONSTRAINT fk_commits_repo    {FK_REPO},
  CONSTRAINT fk_commits_contrib {FK_CONTRIB},
  CONSTRAINT fk_commits_date    {fk_date('commit_date_key')}
) USING DELTA
CLUSTER BY (repository_key, commit_date_key)
''')

run(f'''
CREATE TABLE IF NOT EXISTS {G}.fact_issues (
  record_key        STRING NOT NULL,
  issue_number      BIGINT,
  repository_key    BIGINT NOT NULL,
  contributor_key   BIGINT NOT NULL,
  created_date_key  INT NOT NULL,
  closed_date_key   INT NOT NULL,
  state             STRING,
  is_closed         BOOLEAN,
  hours_to_close    DOUBLE,
  created_at        TIMESTAMP,
  closed_at         TIMESTAMP,
  updated_at        TIMESTAMP,
  batch_id          STRING,
  load_timestamp    TIMESTAMP,
  CONSTRAINT pk_fact_issues PRIMARY KEY (record_key),
  CONSTRAINT fk_issues_repo     {FK_REPO},
  CONSTRAINT fk_issues_contrib  {FK_CONTRIB},
  CONSTRAINT fk_issues_created  {fk_date('created_date_key')},
  CONSTRAINT fk_issues_closed   {fk_date('closed_date_key')}
) USING DELTA
CLUSTER BY (repository_key, created_date_key)
''')

run(f'''
CREATE TABLE IF NOT EXISTS {G}.fact_pull_requests (
  record_key        STRING NOT NULL,
  pr_number         BIGINT,
  repository_key    BIGINT NOT NULL,
  contributor_key   BIGINT NOT NULL,
  created_date_key  INT NOT NULL,
  closed_date_key   INT NOT NULL,
  merged_date_key   INT NOT NULL,
  state             STRING,
  is_draft          BOOLEAN,
  is_merged         BOOLEAN,
  hours_to_merge    DOUBLE,
  hours_to_close    DOUBLE,
  created_at        TIMESTAMP,
  closed_at         TIMESTAMP,
  merged_at         TIMESTAMP,
  batch_id          STRING,
  load_timestamp    TIMESTAMP,
  CONSTRAINT pk_fact_pull_requests PRIMARY KEY (record_key),
  CONSTRAINT fk_pulls_repo     {FK_REPO},
  CONSTRAINT fk_pulls_contrib  {FK_CONTRIB},
  CONSTRAINT fk_pulls_created  {fk_date('created_date_key')},
  CONSTRAINT fk_pulls_closed   {fk_date('closed_date_key')},
  CONSTRAINT fk_pulls_merged   {fk_date('merged_date_key')}
) USING DELTA
CLUSTER BY (repository_key, created_date_key)
''')

run(f'''
CREATE TABLE IF NOT EXISTS {G}.fact_repo_daily_activity (
  repository_key       BIGINT NOT NULL,
  date_key             INT NOT NULL,
  commits              INT,
  issues_opened        INT,
  issues_closed        INT,
  net_issue_change     INT,
  prs_opened           INT,
  prs_merged           INT,
  prs_closed_unmerged  INT,
  rows_aggregated      INT,
  load_timestamp       TIMESTAMP,
  CONSTRAINT pk_fact_repo_daily_activity PRIMARY KEY (repository_key, date_key),
  CONSTRAINT fk_daily_repo {FK_REPO},
  CONSTRAINT fk_daily_date {fk_date('date_key')}
) USING DELTA
CLUSTER BY (repository_key, date_key)
''')

run(f'''
CREATE TABLE IF NOT EXISTS {G}.fact_contributor_activity (
  repository_key    BIGINT NOT NULL,
  contributor_key   BIGINT NOT NULL,
  date_key          INT NOT NULL,
  commits           INT,
  issues_opened     INT,
  prs_opened        INT,
  total_activities  INT,
  load_timestamp    TIMESTAMP,
  CONSTRAINT pk_fact_contributor_activity PRIMARY KEY (repository_key, contributor_key, date_key),
  CONSTRAINT fk_contribact_repo    {FK_REPO},
  CONSTRAINT fk_contribact_contrib {FK_CONTRIB},
  CONSTRAINT fk_contribact_date    {fk_date('date_key')}
) USING DELTA
CLUSTER BY (repository_key, date_key)
''')

# Tiny table (one row per repo per day): clustering is not worth it.
run(f'''
CREATE TABLE IF NOT EXISTS {G}.fact_repo_snapshot (
  repository_key     BIGINT NOT NULL,
  snapshot_date_key  INT NOT NULL,
  stars_count        BIGINT,
  forks_count        BIGINT,
  open_issues_count  BIGINT,
  subscribers_count  BIGINT,
  snapshot_at        TIMESTAMP,
  load_timestamp     TIMESTAMP,
  CONSTRAINT pk_fact_repo_snapshot PRIMARY KEY (repository_key, snapshot_date_key),
  CONSTRAINT fk_snapshot_repo {FK_REPO},
  CONSTRAINT fk_snapshot_date {fk_date('snapshot_date_key')}
) USING DELTA
''')
print('Fact tables ready')

In [0]:
run(f'''
MERGE INTO {G}.dim_date t
USING (
  SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT)        AS date_key,
    d                                              AS full_date,
    year(d)                                        AS year,
    quarter(d)                                     AS quarter,
    month(d)                                       AS month_num,
    date_format(d, 'MMMM')                         AS month_name,
    date_format(d, 'yyyy-MM')                      AS year_month,
    trunc(d, 'MM')                                 AS month_start_date,
    weekofyear(d)                                  AS week_of_year,
    ((dayofweek(d) + 5) % 7) + 1                   AS day_of_week_num,
    date_format(d, 'EEEE')                         AS day_name,
    (((dayofweek(d) + 5) % 7) + 1) >= 6            AS is_weekend
  FROM (SELECT explode(sequence(DATE'2008-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d)
) s
ON t.date_key = s.date_key
WHEN NOT MATCHED THEN INSERT *
''')

# "Not applicable" date member (used for open issues / unmerged PRs)
run(f'''
INSERT INTO {G}.dim_date
SELECT -1, CAST(NULL AS DATE), CAST(NULL AS INT), CAST(NULL AS INT), CAST(NULL AS INT),
       'Not applicable', 'N/A', CAST(NULL AS DATE), CAST(NULL AS INT), CAST(NULL AS INT),
       'Not applicable', CAST(NULL AS BOOLEAN)
WHERE NOT EXISTS (SELECT 1 FROM {G}.dim_date WHERE date_key = -1)
''')

# "Unknown author" contributor member (commits whose email is not linked to a GitHub account)
run(f'''
INSERT INTO {G}.dim_contributor
SELECT -1, 'Unknown', false, CAST(NULL AS DATE), current_timestamp()
WHERE NOT EXISTS (SELECT 1 FROM {G}.dim_contributor WHERE contributor_key = -1)
''')
print('dim_date and unknown members populated')

In [0]:
SILVER_ENTITIES = ['commits', 'issues', 'pulls', 'repo_metadata', 'contributors']
epoch = datetime(1970, 1, 1)
rows = [(repo, entity, 'GOLD', epoch, 'init', now())
        for repo in REPOS for entity in SILVER_ENTITIES]

(DeltaTable.forName(spark, WM_TABLE).alias('t')
    .merge(spark.createDataFrame(rows, WM_SCHEMA).alias('s'),
           't.repository = s.repository AND t.entity = s.entity AND t.layer = s.layer')
    .whenNotMatchedInsertAll()
    .execute())
print('GOLD watermarks initialised (existing rows are never rewound)')

In [0]:
run(f'''
CREATE OR REPLACE VIEW {G}.vw_pipeline_runs AS
SELECT
  execution_id, batch_id, notebook_name, layer, entity_type, repository,
  load_type, run_date, start_time, end_time, status,
  CAST(unix_timestamp(end_time) - unix_timestamp(start_time) AS BIGINT) AS duration_seconds,
  rows_processed, inserted_rows, updated_rows, upserted_rows, quarantined_rows,
  error_message
FROM {CATALOG}.ops.pipeline_execution_logs
''')

run(f'''
CREATE OR REPLACE VIEW {G}.vw_quarantine_summary AS
SELECT 'BRONZE' AS layer, entity, repository, error_type,
       CAST(quarantined_at AS DATE) AS quarantined_date, COUNT(*) AS quarantined_rows
FROM {CATALOG}.ops.bronze_quarantine
GROUP BY entity, repository, error_type, CAST(quarantined_at AS DATE)
UNION ALL
SELECT 'SILVER' AS layer, entity, repository, error_type,
       CAST(quarantined_at AS DATE) AS quarantined_date, COUNT(*) AS quarantined_rows
FROM {CATALOG}.ops.silver_quarantine
GROUP BY entity, repository, error_type, CAST(quarantined_at AS DATE)
''')
print('Views ready')

In [0]:
TABLES = ['dim_date', 'dim_repository', 'dim_contributor', 'fact_commits', 'fact_issues',
          'fact_pull_requests', 'fact_repo_daily_activity', 'fact_contributor_activity',
          'fact_repo_snapshot']

print('=== Tables, clustering columns and row counts ===')
for name in TABLES:
    detail = run(f'DESCRIBE DETAIL {G}.{name}').first()
    count = spark.table(f'{G}.{name}').count()
    print(f'{name:28} clustering={list(detail["clusteringColumns"])!s:38} rows={count:,}')

print('\n=== Primary and foreign keys ===')
display(run(f'''
  SELECT table_name, constraint_name, constraint_type
  FROM {CATALOG}.information_schema.table_constraints
  WHERE table_schema = 'gold'
  ORDER BY table_name, constraint_type, constraint_name
'''))

In [0]:
# Hard checks
assert spark.table(f'{G}.dim_date').count() == 8402, 'dim_date should hold 8,401 days plus the -1 row'
assert spark.table(f'{G}.dim_date').filter('date_key = -1').count() == 1, 'Missing date unknown member'
assert spark.table(f'{G}.dim_contributor').filter('contributor_key = -1').count() == 1, 'Missing contributor unknown member'
assert spark.table(WM_TABLE).filter("layer = 'GOLD'").count() == len(REPOS) * len(SILVER_ENTITIES), 'Unexpected GOLD watermark count'

# Day-of-week sanity check: 2026-10-24 is a Saturday (ISO day 6, weekend)
check = spark.table(f'{G}.dim_date').filter('date_key = 20261024').first()
assert (check['day_name'], check['day_of_week_num'], check['is_weekend']) == ('Saturday', 6, True), check

print('All Gold setup checks passed')
display(spark.table(f'{G}.vw_pipeline_runs').limit(5))